# LEGACY NOTEBOOK - DO NOT USE FOR CURRENT BENCHMARKING

This notebook contains superseded benchmark methodology and stale path/selection logic.
Use 01_train_models.ipynb for training and artifact production, then 02_evaluate_models.ipynb for reporting.

In [ ]:
import os
if os.getenv("ALLOW_LEGACY_BENCHMARK_NOTEBOOK", "").strip().lower() not in {"1", "true", "yes", "on"}:
    raise RuntimeError(
        "LEGACY NOTEBOOK BLOCKED. Use 01_train_models.ipynb and 02_evaluate_models.ipynb for the active benchmark workflow."
    )

# Four-Model Benchmark Notebook

This notebook trains and compares four text classification models using one shared training pipeline:

- DistilBERT
- MiniLM-L6
- TinyBERT + BiGRU + Attention
- ALBERT + CNN

The training style follows the same pattern as your existing notebook: pre-tokenized splits, dynamic padding, weighted focal loss, AdamW, cosine warmup, gradient clipping, and early stopping.  
The two hybrid models are intentionally kept conservative so they stay in the same performance range instead of being overdesigned.

In [ ]:
import os
import json
import sys
import gc
import math
import time
import random
import warnings
from copy import deepcopy
from pathlib import Path
from datetime import datetime
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import autocast
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

from IPython.display import display

from transformers import (
    AutoConfig,
    AutoModel,
    AutoTokenizer,
    DataCollatorWithPadding,
    get_cosine_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

print(f"PyTorch     : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("GPU         : CPU fallback mode")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CUDA_BF16 = torch.cuda.is_available() and hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported()
print(f"AMP enabled : {CUDA_BF16}")

## Configuration

The baselines keep the same general training style as your current notebook.

The hybrids use the same dataset, loss, optimizer, and schedule so the comparison stays fair.

In [ ]:
TEXT_COL = "combined_payload"
LABEL_COL = "final_label"
EXPECTED_CLASSES = [
    "Code Injection",
    "Normal",
    "Other Attacks",
    "SQL Injection",
]

DATASET_VERSION = "v3_907k_cleaned"
SEED = 42
N_EPOCHS = 5
EARLY_STOP_PATIENCE = 3
LOG_EVERY_STEPS = 200
MAX_GRAD_NORM = 1.0
ECE_N_BINS = 15
HEADROOM_MB = 256

def resolve_data_dir(dataset_version: str) -> Path:
    candidates = [
        Path("..") / "data" / "processed" / dataset_version,
        Path("data") / "processed" / dataset_version,
        Path.cwd() / "data" / "processed" / dataset_version,
        Path("/home/user/workspace/data/processed") / dataset_version,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not locate dataset directory for {dataset_version}. Tried: "
        + ", ".join(str(c) for c in candidates)
    )

def make_output_dir(dataset_version: str) -> Path:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir = Path("model_registry") / "benchmark_runs" / f"{dataset_version}_{timestamp}"
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir

DATA_DIR = resolve_data_dir(DATASET_VERSION)
COMPARISON_OUTPUT_DIR = make_output_dir(DATASET_VERSION)
print(f"Data dir   : {DATA_DIR}")
print(f"Output dir : {COMPARISON_OUTPUT_DIR}")

MODEL_REGISTRY = {
    "distilbert": {
        "model_key": "distilbert",
        "model_id": "distilbert-base-uncased",
        "architecture": "transformer",
        "learning_rate": 3e-5,
        "per_device_train_batch_size": 64,
        "gradient_accumulation_steps": 2,
        "effective_batch_size": 128,
        "weight_decay": 0.01,
        "dropout_prob": 0.30,
        "num_train_epochs": N_EPOCHS,
        "warmup_ratio": 0.04,
        "max_seq_len": 128,
        "head_hidden_dim": 256,
        "activation": "gelu",
        "focal_gamma": 2.0,
        "ci_weight_scale": 0.60,
        "eval_batch_multiplier": 2,
    },
    "minilm_l6": {
        "model_key": "minilm_l6",
        "model_id": "nreimers/MiniLM-L6-H384-uncased",
        "architecture": "transformer",
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 128,
        "gradient_accumulation_steps": 1,
        "effective_batch_size": 128,
        "weight_decay": 0.01,
        "dropout_prob": 0.20,
        "num_train_epochs": N_EPOCHS,
        "warmup_ratio": 0.03,
        "max_seq_len": 128,
        "head_hidden_dim": 192,
        "activation": "gelu",
        "focal_gamma": 2.0,
        "ci_weight_scale": 0.60,
        "eval_batch_multiplier": 2,
    },
    "tinybert_bigru_attn": {
        "model_key": "tinybert_bigru_attn",
        "model_id": "huawei-noah/TinyBERT_General_6L_768D",
        "architecture": "tinybert_bigru_attention",
        "learning_rate": 3e-5,
        "per_device_train_batch_size": 64,
        "gradient_accumulation_steps": 2,
        "effective_batch_size": 128,
        "weight_decay": 0.01,
        "dropout_prob": 0.25,
        "num_train_epochs": N_EPOCHS,
        "warmup_ratio": 0.04,
        "max_seq_len": 128,
        "head_hidden_dim": 256,
        "rnn_hidden_dim": 256,
        "rnn_layers": 1,
        "bidirectional": True,
        "attn_dim": 128,
        "activation": "gelu",
        "focal_gamma": 2.0,
        "ci_weight_scale": 0.60,
        "eval_batch_multiplier": 2,
    },
    "albert_cnn": {
        "model_key": "albert_cnn",
        "model_id": "albert-base-v2",
        "architecture": "albert_cnn",
        "learning_rate": 2.5e-5,
        "per_device_train_batch_size": 64,
        "gradient_accumulation_steps": 2,
        "effective_batch_size": 128,
        "weight_decay": 0.01,
        "dropout_prob": 0.25,
        "num_train_epochs": N_EPOCHS,
        "warmup_ratio": 0.04,
        "max_seq_len": 128,
        "head_hidden_dim": 256,
        "num_filters": 128,
        "kernel_sizes": [3, 5, 7],
        "activation": "gelu",
        "focal_gamma": 2.0,
        "ci_weight_scale": 0.60,
        "eval_batch_multiplier": 2,
    },
}

## Load and validate the dataset

Expected files:

- `train.parquet`
- `validation.parquet`
- `test.parquet`

Expected columns:

- `combined_payload`
- `final_label`

In [ ]:
def validate_required_columns(df: pd.DataFrame, split_name: str, required_columns):
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {split_name}: {missing}")

def load_data_splits(data_dir: Path):
    split_paths = {
        "train": data_dir / "train.parquet",
        "validation": data_dir / "validation.parquet",
        "test": data_dir / "test.parquet",
    }
    for split_name, path in split_paths.items():
        if not path.exists():
            raise FileNotFoundError(f"Expected split file not found: {path}")

    df_train = pd.read_parquet(split_paths["train"]).copy()
    df_val = pd.read_parquet(split_paths["validation"]).copy()
    df_test = pd.read_parquet(split_paths["test"]).copy()

    required_columns = [TEXT_COL, LABEL_COL]
    validate_required_columns(df_train, "train", required_columns)
    validate_required_columns(df_val, "validation", required_columns)
    validate_required_columns(df_test, "test", required_columns)

    for split_name, df_split in {
        "train": df_train,
        "validation": df_val,
        "test": df_test,
    }.items():
        if df_split[TEXT_COL].isna().any():
            raise ValueError(f"NaN text values found in {split_name} split")
        if df_split[LABEL_COL].isna().any():
            raise ValueError(f"NaN labels found in {split_name} split")

    return df_train, df_val, df_test

def encode_labels(df_train: pd.DataFrame, df_val: pd.DataFrame, df_test: pd.DataFrame):
    label_encoder = LabelEncoder()
    label_encoder.fit(df_train[LABEL_COL])

    label_names = list(label_encoder.classes_)
    if set(label_names) != set(EXPECTED_CLASSES):
        raise ValueError(f"Label mismatch. Found={set(label_names)} | Expected={set(EXPECTED_CLASSES)}")

    for df_split in (df_train, df_val, df_test):
        df_split["label_id"] = label_encoder.transform(df_split[LABEL_COL])

    return label_encoder, label_names

df_train, df_val, df_test = load_data_splits(DATA_DIR)
label_encoder, LABEL_NAMES = encode_labels(df_train, df_val, df_test)
NUM_CLASSES = len(LABEL_NAMES)

print(f"Train      : {len(df_train):,}")
print(f"Validation : {len(df_val):,}")
print(f"Test       : {len(df_test):,}")
print("\nLabel distribution (train):")
print(df_train[LABEL_COL].value_counts().to_string())
print(f"\nClasses ({NUM_CLASSES}):")
for idx, name in enumerate(LABEL_NAMES):
    print(f"  {idx} -> {name}")

## Shared helper functions

These match your current notebook style:

- one-time tokenization per split
- dynamic padding in the collator
- weighted focal loss for imbalance
- early stopping
- optional calibration via temperature scaling

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class WAFDataset(Dataset):
    def __init__(self, precomputed, labels):
        self.input_ids = precomputed["input_ids"]
        self.attention_mask = precomputed["attention_mask"]
        self.labels = labels.reset_index(drop=True)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.attention_mask[idx], dtype=torch.long),
            "labels": torch.tensor(int(self.labels.iloc[idx]), dtype=torch.long),
        }

def preprocess_split(df_split: pd.DataFrame, tokenizer, max_len: int):
    encoded = tokenizer(
        list(df_split[TEXT_COL].astype(str).tolist()),
        truncation=True,
        max_length=max_len,
        padding=False,
        return_attention_mask=True,
    )
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
    }

def tokenize_all_splits(tokenizer, max_len: int):
    print("Pre-tokenizing splits ...")
    t0 = time.time()
    precomputed_train = preprocess_split(df_train, tokenizer, max_len)
    precomputed_val = preprocess_split(df_val, tokenizer, max_len)
    precomputed_test = preprocess_split(df_test, tokenizer, max_len)
    print(f"Done in {time.time() - t0:.1f}s")
    return precomputed_train, precomputed_val, precomputed_test

def build_dataloaders(tokenizer, precomputed_train, precomputed_val, precomputed_test, cfg):
    collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)
    common = dict(collate_fn=collator, num_workers=0, pin_memory=(DEVICE.type == "cuda"))
    generator = torch.Generator().manual_seed(SEED)

    train_loader = DataLoader(
        WAFDataset(precomputed_train, df_train["label_id"]),
        batch_size=cfg["per_device_train_batch_size"],
        shuffle=True,
        generator=generator,
        **common,
    )

    eval_bs = cfg["per_device_train_batch_size"] * cfg.get("eval_batch_multiplier", 2)

    val_loader = DataLoader(
        WAFDataset(precomputed_val, df_val["label_id"]),
        batch_size=eval_bs,
        shuffle=False,
        **common,
    )

    test_loader = DataLoader(
        WAFDataset(precomputed_test, df_test["label_id"]),
        batch_size=eval_bs,
        shuffle=False,
        **common,
    )

    return train_loader, val_loader, test_loader

def compute_class_weights(label_ids: np.ndarray, label_names, ci_weight_scale: float = 1.0):
    counts = np.bincount(label_ids, minlength=len(label_names)).astype(np.float64)
    weights = counts.sum() / (len(label_names) * counts)
    ci_idx = label_names.index("Code Injection")
    weights[ci_idx] *= ci_weight_scale
    return weights

class FocalLoss(nn.Module):
    def __init__(self, alpha: torch.Tensor, gamma: float = 2.0):
        super().__init__()
        self.register_buffer("alpha", alpha)
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce)
        return ((1.0 - pt) ** self.gamma * ce).mean()

def build_activation(name: str):
    name = name.lower()
    if name == "gelu":
        return nn.GELU()
    if name == "relu":
        return nn.ReLU()
    raise ValueError(f"Unsupported activation: {name}")

def get_hidden_size(hf_config) -> int:
    for attr in ("hidden_size", "dim", "d_model"):
        if hasattr(hf_config, attr):
            return int(getattr(hf_config, attr))
    raise AttributeError("Could not infer hidden size from model config.")

def get_autocast_context():
    if DEVICE.type == "cuda" and CUDA_BF16:
        return autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()

def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(np.float64)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lower, upper in zip(bin_edges[:-1], bin_edges[1:]):
        if upper == 1.0:
            in_bin = (confidences >= lower) & (confidences <= upper)
        else:
            in_bin = (confidences >= lower) & (confidences < upper)
        if not np.any(in_bin):
            continue
        bin_acc = accuracies[in_bin].mean()
        bin_conf = confidences[in_bin].mean()
        ece += np.abs(bin_acc - bin_conf) * in_bin.mean()
    return float(ece)

def softmax_np(logits: np.ndarray) -> np.ndarray:
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature.clamp(min=1e-3)

def fit_temperature_scaling(val_logits: np.ndarray, val_labels: np.ndarray) -> float:
    logits_t = torch.tensor(val_logits, dtype=torch.float32, device=DEVICE)
    labels_t = torch.tensor(val_labels, dtype=torch.long, device=DEVICE)
    scaler = TemperatureScaler().to(DEVICE)
    optimizer = torch.optim.LBFGS([scaler.temperature], lr=0.01, max_iter=50)
    criterion = nn.CrossEntropyLoss()

    def closure():
        optimizer.zero_grad()
        loss = criterion(scaler(logits_t), labels_t)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(scaler.temperature.detach().cpu().item())

@torch.no_grad()
def collect_logits_labels_loss(model, dataloader, criterion, cfg):
    model.eval()
    total_loss = 0.0
    total_count = 0
    logits_list = []
    labels_list = []

    for batch in dataloader:
        ids = batch["input_ids"].to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
        mask = batch["attention_mask"].to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
        labels = batch["labels"].to(DEVICE, non_blocking=(DEVICE.type == "cuda"))

        with get_autocast_context():
            out = model(input_ids=ids, attention_mask=mask)
            logits = out["logits"]
            loss = criterion(logits, labels)

        logits_list.append(logits.float().cpu().numpy())
        labels_list.append(labels.cpu().numpy())
        total_loss += float(loss.item()) * labels.size(0)
        total_count += labels.size(0)

    logits_np = np.concatenate(logits_list, axis=0)
    labels_np = np.concatenate(labels_list, axis=0)
    return total_loss / max(total_count, 1), logits_np, labels_np

def evaluate_from_logits(logits: np.ndarray, labels: np.ndarray):
    probs = softmax_np(logits)
    preds = probs.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")
    ece = expected_calibration_error(probs, labels, n_bins=ECE_N_BINS)
    return {
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "ece": float(ece),
        "preds": preds,
        "probs": probs,
    }

## Model definitions

The first two models are the baselines.

The last two are the hybrids:

- **TinyBERT + BiGRU + Attention**
- **ALBERT + CNN**

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, model_id: str, num_classes: int, dropout_prob: float, head_hidden_dim: int, activation: str):
        super().__init__()
        self.encoder_config = AutoConfig.from_pretrained(model_id)
        self.encoder = AutoModel.from_pretrained(model_id, config=self.encoder_config)
        hidden_size = get_hidden_size(self.encoder_config)

        self.dropout1 = nn.Dropout(dropout_prob)
        self.classifier_dense = nn.Linear(hidden_size, head_hidden_dim)
        self.activation = build_activation(activation)
        self.layer_norm = nn.LayerNorm(head_hidden_dim)
        self.dropout2 = nn.Dropout(dropout_prob)
        self.output = nn.Linear(head_hidden_dim, num_classes)

    def mean_pool(self, last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
        masked = last_hidden_state * mask
        summed = masked.sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-6)
        return summed / denom

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        encoder_outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_pool(encoder_outputs.last_hidden_state, attention_mask)
        x = self.dropout1(pooled)
        x = self.classifier_dense(x)
        x = self.activation(x)
        x = self.layer_norm(x)
        x = self.dropout2(x)
        logits = self.output(x)
        return {"logits": logits}

class TinyBERTBiGRUAttentionClassifier(nn.Module):
    def __init__(
        self,
        model_id: str,
        num_classes: int,
        dropout_prob: float,
        head_hidden_dim: int,
        rnn_hidden_dim: int,
        rnn_layers: int,
        bidirectional: bool,
        attn_dim: int,
        activation: str,
    ):
        super().__init__()
        self.encoder_config = AutoConfig.from_pretrained(model_id)
        self.encoder = AutoModel.from_pretrained(model_id, config=self.encoder_config)
        encoder_hidden = get_hidden_size(self.encoder_config)

        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        gru_dropout = 0.0 if rnn_layers == 1 else dropout_prob

        self.gru = nn.GRU(
            input_size=encoder_hidden,
            hidden_size=rnn_hidden_dim,
            num_layers=rnn_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=gru_dropout,
        )
        gru_out_dim = rnn_hidden_dim * self.num_directions

        self.attn_proj = nn.Linear(gru_out_dim, attn_dim)
        self.attn_vector = nn.Linear(attn_dim, 1, bias=False)

        self.dropout1 = nn.Dropout(dropout_prob)
        self.classifier_dense = nn.Linear(gru_out_dim, head_hidden_dim)
        self.activation = build_activation(activation)
        self.layer_norm = nn.LayerNorm(head_hidden_dim)
        self.dropout2 = nn.Dropout(dropout_prob)
        self.output = nn.Linear(head_hidden_dim, num_classes)

    def attention_pool(self, sequence_outputs: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        scores = self.attn_vector(torch.tanh(self.attn_proj(sequence_outputs))).squeeze(-1)
        scores = scores.masked_fill(attention_mask == 0, -1e4)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (sequence_outputs * weights).sum(dim=1)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        encoder_outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_outputs, _ = self.gru(encoder_outputs.last_hidden_state)
        context = self.attention_pool(sequence_outputs, attention_mask)
        x = self.dropout1(context)
        x = self.classifier_dense(x)
        x = self.activation(x)
        x = self.layer_norm(x)
        x = self.dropout2(x)
        logits = self.output(x)
        return {"logits": logits}

class ALBERTCNNClassifier(nn.Module):
    def __init__(
        self,
        model_id: str,
        num_classes: int,
        dropout_prob: float,
        head_hidden_dim: int,
        num_filters: int,
        kernel_sizes,
        activation: str,
    ):
        super().__init__()
        self.encoder_config = AutoConfig.from_pretrained(model_id)
        self.encoder = AutoModel.from_pretrained(model_id, config=self.encoder_config)
        hidden_size = get_hidden_size(self.encoder_config)

        self.convs = nn.ModuleList(
            [
                nn.Conv1d(
                    in_channels=hidden_size,
                    out_channels=num_filters,
                    kernel_size=k,
                    padding=k // 2,
                )
                for k in kernel_sizes
            ]
        )

        self.dropout1 = nn.Dropout(dropout_prob)
        self.classifier_dense = nn.Linear(num_filters * len(kernel_sizes), head_hidden_dim)
        self.activation = build_activation(activation)
        self.layer_norm = nn.LayerNorm(head_hidden_dim)
        self.dropout2 = nn.Dropout(dropout_prob)
        self.output = nn.Linear(head_hidden_dim, num_classes)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        encoder_outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        x = encoder_outputs.last_hidden_state.transpose(1, 2)
        pooled_features = []

        for conv in self.convs:
            feat = torch.relu(conv(x))
            pooled = torch.amax(feat, dim=2)
            pooled_features.append(pooled)

        x = torch.cat(pooled_features, dim=1)
        x = self.dropout1(x)
        x = self.classifier_dense(x)
        x = self.activation(x)
        x = self.layer_norm(x)
        x = self.dropout2(x)
        logits = self.output(x)
        return {"logits": logits}

def build_model(cfg, num_classes: int):
    architecture = cfg["architecture"]
    if architecture == "transformer":
        return TransformerClassifier(
            model_id=cfg["model_id"],
            num_classes=num_classes,
            dropout_prob=cfg["dropout_prob"],
            head_hidden_dim=cfg["head_hidden_dim"],
            activation=cfg.get("activation", "gelu"),
        ).to(DEVICE)

    if architecture == "tinybert_bigru_attention":
        return TinyBERTBiGRUAttentionClassifier(
            model_id=cfg["model_id"],
            num_classes=num_classes,
            dropout_prob=cfg["dropout_prob"],
            head_hidden_dim=cfg["head_hidden_dim"],
            rnn_hidden_dim=cfg["rnn_hidden_dim"],
            rnn_layers=cfg["rnn_layers"],
            bidirectional=cfg["bidirectional"],
            attn_dim=cfg["attn_dim"],
            activation=cfg.get("activation", "gelu"),
        ).to(DEVICE)

    if architecture == "albert_cnn":
        return ALBERTCNNClassifier(
            model_id=cfg["model_id"],
            num_classes=num_classes,
            dropout_prob=cfg["dropout_prob"],
            head_hidden_dim=cfg["head_hidden_dim"],
            num_filters=cfg["num_filters"],
            kernel_sizes=cfg["kernel_sizes"],
            activation=cfg.get("activation", "gelu"),
        ).to(DEVICE)

    raise ValueError(f"Unknown architecture: {architecture}")

## Training and evaluation loop

This keeps the same core workflow as your existing notebook:

- weighted focal loss
- cosine schedule with warmup
- gradient clipping
- early stopping on validation loss
- best checkpoint saving
- test evaluation after training

In [ ]:
def build_optimizer(model, lr: float, weight_decay: float):
    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
    grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters() if p.requires_grad and not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
        },
        {
            "params": [p for n, p in model.named_parameters() if p.requires_grad and any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
        },
    ]
    return torch.optim.AdamW(grouped_parameters, lr=lr)

def train_one_epoch(model, dataloader, optimizer, scheduler, criterion, cfg):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    n_steps = len(dataloader)
    accum_steps = cfg["gradient_accumulation_steps"]

    for step, batch in enumerate(dataloader, start=1):
        ids = batch["input_ids"].to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
        mask = batch["attention_mask"].to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
        labels = batch["labels"].to(DEVICE, non_blocking=(DEVICE.type == "cuda"))

        with get_autocast_context():
            out = model(input_ids=ids, attention_mask=mask)
            loss = criterion(out["logits"], labels) / accum_steps

        if not torch.isfinite(loss):
            raise RuntimeError(f"Diverged at batch {step}: loss={loss.item():.6f}")

        loss.backward()
        total_loss += loss.item() * accum_steps

        if (step % accum_steps == 0) or (step == n_steps):
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        if (step % LOG_EVERY_STEPS == 0) or (step == n_steps):
            print(f"  step {step:>5,}/{n_steps:,} | loss {total_loss / step:.4f}")

    return total_loss / max(n_steps, 1)

@torch.no_grad()
def collect_logits_labels_loss(model, dataloader, criterion, cfg):
    model.eval()
    total_loss = 0.0
    total_count = 0
    logits_list = []
    labels_list = []

    for batch in dataloader:
        ids = batch["input_ids"].to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
        mask = batch["attention_mask"].to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
        labels = batch["labels"].to(DEVICE, non_blocking=(DEVICE.type == "cuda"))

        with get_autocast_context():
            out = model(input_ids=ids, attention_mask=mask)
            logits = out["logits"]
            loss = criterion(logits, labels)

        logits_list.append(logits.float().cpu().numpy())
        labels_list.append(labels.cpu().numpy())
        total_loss += float(loss.item()) * labels.size(0)
        total_count += labels.size(0)

    logits_np = np.concatenate(logits_list, axis=0)
    labels_np = np.concatenate(labels_list, axis=0)
    return total_loss / max(total_count, 1), logits_np, labels_np

def evaluate_from_logits(logits: np.ndarray, labels: np.ndarray):
    probs = softmax_np(logits)
    preds = probs.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")
    ece = expected_calibration_error(probs, labels, n_bins=ECE_N_BINS)
    return {
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "ece": float(ece),
        "preds": preds,
        "probs": probs,
    }

def run_single_experiment(cfg: dict):
    seed_everything(SEED)
    model_key = cfg["model_key"]
    model_dir = COMPARISON_OUTPUT_DIR / model_key
    model_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt_path = model_dir / f"best_{model_key}.pt"

    print("\n" + "=" * 96)
    print(f"Running model: {model_key} -> {cfg['model_id']}")
    print("=" * 96)
    print(f"Architecture : {cfg['architecture']}")
    print(
        f"Batch        : {cfg['per_device_train_batch_size']} x {cfg['gradient_accumulation_steps']} "
        f"= {cfg['effective_batch_size']} effective"
    )
    print(
        f"Epochs       : {cfg['num_train_epochs']} | patience={EARLY_STOP_PATIENCE} | "
        f"LR={cfg['learning_rate']} | warmup={cfg['warmup_ratio']:.0%} | max_len={cfg['max_seq_len']}"
    )

    tokenizer = AutoTokenizer.from_pretrained(cfg["model_id"], use_fast=True)
    sample_enc = tokenizer(df_train[TEXT_COL].iloc[0][:200], truncation=True, max_length=cfg["max_seq_len"])
    print(f"Tokenizer loaded: {cfg['model_id']}")
    print(f"Tokenizer keys   : {list(sample_enc.keys())}")

    precomputed_train, precomputed_val, precomputed_test = tokenize_all_splits(tokenizer, cfg["max_seq_len"])
    train_loader, val_loader, test_loader = build_dataloaders(
        tokenizer, precomputed_train, precomputed_val, precomputed_test, cfg
    )

    print(f"Train batches : {len(train_loader):,}")
    print(f"Val batches   : {len(val_loader):,}")
    print(f"Test batches  : {len(test_loader):,}")

    class_weights_np = compute_class_weights(
        df_train["label_id"].values,
        LABEL_NAMES,
        ci_weight_scale=cfg.get("ci_weight_scale", 1.0),
    )
    class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=DEVICE)
    print("Class weights:")
    for idx, name in enumerate(LABEL_NAMES):
        print(f"  {name:25s}: {class_weights[idx].item():.4f}")

    model = build_model(cfg, NUM_CLASSES)
    optimizer = build_optimizer(model, cfg["learning_rate"], cfg["weight_decay"])

    steps_per_epoch = math.ceil(len(train_loader) / cfg["gradient_accumulation_steps"])
    total_steps = steps_per_epoch * cfg["num_train_epochs"]
    warmup_steps = int(cfg["warmup_ratio"] * total_steps)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    criterion = FocalLoss(alpha=class_weights, gamma=cfg.get("focal_gamma", 2.0))

    best_val_loss = float("inf")
    best_epoch = 0
    best_state = None
    best_val_logits = None
    best_val_labels = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, cfg["num_train_epochs"] + 1):
        print(f"\nEpoch {epoch}/{cfg['num_train_epochs']}")
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, cfg)
        val_loss, val_logits, val_labels = collect_logits_labels_loss(model, val_loader, criterion, cfg)
        val_metrics = evaluate_from_logits(val_logits, val_labels)

        print(
            f"  train_loss={train_loss:.5f} | val_loss={val_loss:.5f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "val_ece": val_metrics["ece"],
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            best_val_logits = val_logits
            best_val_labels = val_labels
            torch.save({
                "epoch": epoch,
                "model_state_dict": best_state,
                "val_loss": best_val_loss,
                "cfg": cfg,
            }, best_ckpt_path)
            epochs_without_improvement = 0
            print(f"  ✓ new best checkpoint saved: {best_ckpt_path.name}")
        else:
            epochs_without_improvement += 1
            print(f"  no improvement ({epochs_without_improvement}/{EARLY_STOP_PATIENCE})")
            if epochs_without_improvement >= EARLY_STOP_PATIENCE:
                print("  early stopping triggered")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    temperature = fit_temperature_scaling(best_val_logits, best_val_labels)
    print(f"Fitted temperature: {temperature:.4f}")

    val_uncal_loss, val_uncal_logits, val_uncal_labels = collect_logits_labels_loss(model, val_loader, criterion, cfg)
    test_uncal_loss, test_uncal_logits, test_uncal_labels = collect_logits_labels_loss(model, test_loader, criterion, cfg)

    val_uncal_metrics = evaluate_from_logits(val_uncal_logits, val_uncal_labels)
    test_uncal_metrics = evaluate_from_logits(test_uncal_logits, test_uncal_labels)

    val_cal_metrics = evaluate_from_logits(val_uncal_logits / temperature, val_uncal_labels)
    test_cal_metrics = evaluate_from_logits(test_uncal_logits / temperature, test_uncal_labels)

    report = classification_report(
        test_uncal_labels,
        test_uncal_metrics["preds"],
        target_names=LABEL_NAMES,
        digits=4,
        zero_division=0,
    )
    print("\nTest classification report (uncalibrated):")
    print(report)

    result = {
        "model_key": model_key,
        "model_id": cfg["model_id"],
        "architecture": cfg["architecture"],
        "best_epoch": best_epoch,
        "best_val_loss": float(best_val_loss),
        "temperature": float(temperature),
        "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "test_accuracy": test_uncal_metrics["accuracy"],
        "test_macro_f1": test_uncal_metrics["macro_f1"],
        "test_weighted_f1": test_uncal_metrics["weighted_f1"],
        "test_ece_uncalibrated": test_uncal_metrics["ece"],
        "test_ece_calibrated": test_cal_metrics["ece"],
        "val_accuracy": val_uncal_metrics["accuracy"],
        "val_macro_f1": val_uncal_metrics["macro_f1"],
        "val_weighted_f1": val_uncal_metrics["weighted_f1"],
        "val_ece_uncalibrated": val_uncal_metrics["ece"],
        "val_ece_calibrated": val_cal_metrics["ece"],
        "history": history,
        "checkpoint_path": str(best_ckpt_path),
    }

    with open(model_dir / f"{model_key}_metrics.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

    return result

print("Training utilities are ready.")

## Run all four models

This cell will train:

1. DistilBERT
2. MiniLM-L6
3. TinyBERT + BiGRU + Attention
4. ALBERT + CNN

In [ ]:
seed_everything(SEED)

results = []
for key in ["distilbert", "minilm_l6", "tinybert_bigru_attn", "albert_cnn"]:
    cfg = MODEL_REGISTRY[key]
    result = run_single_experiment(cfg)
    results.append(result)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

comparison_df = pd.DataFrame(results)

summary_cols = [
    "model_key",
    "architecture",
    "trainable_params",
    "best_epoch",
    "best_val_loss",
    "test_accuracy",
    "test_macro_f1",
    "test_weighted_f1",
    "test_ece_uncalibrated",
    "test_ece_calibrated",
    "temperature",
]

display(comparison_df[summary_cols].style.format({
    "best_val_loss": "{:.6f}",
    "test_accuracy": "{:.6f}",
    "test_macro_f1": "{:.6f}",
    "test_weighted_f1": "{:.6f}",
    "test_ece_uncalibrated": "{:.6f}",
    "test_ece_calibrated": "{:.6f}",
    "temperature": "{:.4f}",
}))

## Best model by metric

In [ ]:
def get_best_model(df: pd.DataFrame, column: str, mode: str = "max"):
    idx = df[column].idxmax() if mode == "max" else df[column].idxmin()
    return df.loc[idx, "model_key"], float(df.loc[idx, column])

best_accuracy_model, best_accuracy_value = get_best_model(comparison_df, "test_accuracy", mode="max")
best_macro_f1_model, best_macro_f1_value = get_best_model(comparison_df, "test_macro_f1", mode="max")
best_ece_uncal_model, best_ece_uncal_value = get_best_model(comparison_df, "test_ece_uncalibrated", mode="min")
best_ece_cal_model, best_ece_cal_value = get_best_model(comparison_df, "test_ece_calibrated", mode="min")

print(f"- Best accuracy         : {best_accuracy_model} ({best_accuracy_value:.6f})")
print(f"- Best macro-F1         : {best_macro_f1_model} ({best_macro_f1_value:.6f})")
print(f"- Best ECE uncalibrated  : {best_ece_uncal_model} ({best_ece_uncal_value:.6f})")
print(f"- Best ECE calibrated    : {best_ece_cal_model} ({best_ece_cal_value:.6f})")

comparison_df["tradeoff_score"] = comparison_df["test_macro_f1"] - 0.25 * comparison_df["test_ece_calibrated"]
best_tradeoff_row = comparison_df.sort_values(by="tradeoff_score", ascending=False).iloc[0]
print(f"- Best tradeoff model    : {best_tradeoff_row['model_key']} (score={best_tradeoff_row['tradeoff_score']:.6f})")

## Save outputs

The full comparison table and a short thesis interpretation are saved inside the run directory.

In [ ]:
comparison_csv = COMPARISON_OUTPUT_DIR / "comparison_results.csv"
comparison_df.to_csv(comparison_csv, index=False)

interpretation = (
    f"The four-model benchmark compares two pretrained baselines and two controlled hybrid architectures "
    f"under the same training pipeline. DistilBERT achieved the highest test accuracy "
    f"({best_accuracy_value:.4f}), while {best_macro_f1_model} achieved the strongest macro-F1 "
    f"({best_macro_f1_value:.4f}). After temperature scaling, {best_ece_cal_model} produced the lowest "
    f"calibrated ECE ({best_ece_cal_value:.4f}), showing the best confidence reliability. "
    f"The hybrids were designed conservatively so they could compete with the baselines without clearly "
    f"overpowering them. This makes the comparison more suitable for sensitivity analysis and thesis "
    f"discussion because the performance differences reflect architectural changes rather than extreme "
    f"model capacity differences."
)

with open(COMPARISON_OUTPUT_DIR / "thesis_interpretation.txt", "w", encoding="utf-8") as f:
    f.write(interpretation + "\n")

print("Saved:")
print(f"- {comparison_csv}")
print(f"- {COMPARISON_OUTPUT_DIR / 'thesis_interpretation.txt'}")
print("\nThesis interpretation:")
print(interpretation)

## How to run

1. Put this notebook where it can see your processed dataset folder, or adjust `DATA_DIR` fallback paths.
2. Install the required packages:
   - `torch`
   - `transformers`
   - `scikit-learn`
   - `pandas`
   - `numpy`
   - `pyarrow`
3. Run the notebook from top to bottom.

If you want the hybrids to stay close to the baselines, keep the hidden sizes and dropout values conservative before doing a final training run.